In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17968, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 17968 (delta 95), reused 51 (delta 51), pack-reused 17849 (from 3)
Receiving objects: 100% (17968/17968), 17.10 MiB | 16.62 MiB/s, done.
Resolving deltas: 100% (12226/12226), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 1.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:

weights = "/content/drive/MyDrive/walnut/best.pt"

In [ ]:

!pip install filterpy
!pip install lap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=d63d65692d7c21273d82b0b171bb0f91bd510f8f8e3a71f20235735c6ad1b6de
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 6.3 MB/s eta 0:00:00


In [ ]:

!git clone https://github.com/abewley/sort.git

Cloning into 'sort'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 208 (delta 45), reused 40 (delta 40), pack-reused 159 (from 1)
Receiving objects: 100% (208/208), 1.20 MiB | 5.53 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [ ]:

!sed -i "s/TkAgg/Agg/g" sort/sort.py

In [ ]:

import importlib.util

spec = importlib.util.spec_from_file_location("sort_module", "/content/yolov5/sort/sort.py")
sort_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sort_module)

tracker = sort_module.Sort()
print("SORT loaded ✅")

SORT loaded ✅


In [ ]:

!ls -la /content/yolov5/sort
!python -c "import sys; sys.path.insert(0,'/content/yolov5/sort'); import sort; print(sort.__file__); print(dir(sort)[:30])"

total 84
drwxr-xr-x  5 root root  4096 May 19 11:19 .
drwxr-xr-x 11 root root  4096 May 19 11:18 ..
drwxr-xr-x  3 root root  4096 May 19 11:18 data
drwxr-xr-x  8 root root  4096 May 19 11:18 .git
-rw-r--r--  1 root root    22 May 19 11:18 .gitignore
-rw-r--r--  1 root root 35141 May 19 11:18 LICENSE
drwxr-xr-x  2 root root  4096 May 19 11:19 __pycache__
-rw-r--r--  1 root root  5416 May 19 11:18 README.md
-rw-r--r--  1 root root    48 May 19 11:18 requirements.txt
-rw-r--r--  1 root root 11737 May 19 11:19 sort.py
/content/yolov5/sort/sort.py
['KalmanBoxTracker', 'KalmanFilter', 'Sort', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'argparse', 'associate_detections_to_trackers', 'convert_bbox_to_z', 'convert_x_to_bbox', 'glob', 'io', 'iou_batch', 'linear_assignment', 'matplotlib', 'np', 'os', 'parse_args', 'patches', 'plt', 'print_function', 'time']


In [ ]:

# ================== IMPORTS ==================
import cv2
import numpy as np
import sys
from ultralytics import YOLO

# ================== SORT ==================
import importlib.util

spec = importlib.util.spec_from_file_location("sort_module", "/content/yolov5/sort/sort.py")
sort_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sort_module)

tracker = sort_module.Sort()
print("✅ SORT loaded via manual import")


# ================== YOLO ==================
weights_path = "/content/drive/MyDrive/walnut/runs/detect/train4/weights/best.pt"
model = YOLO(weights_path)
print("✅ YOLO model loaded")

# ================== VIDEO ==================
video_path = "/content/drive/MyDrive/walnut/Video/walnut Tree.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("❌ Video not opened")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25

out_path = "/content/counted_output.mp4"
out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

# ================== COUNTING ==================
line_y = height // 2
count = 0
counted_ids = set()

print("▶️ Processing video...")

# ================== LOOP ==================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ---------- YOLO inference ----------
    results = model(frame, conf=0.4, iou=0.5, verbose=False)[0]

    detections = []
    if results.boxes is not None:
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            detections.append([x1, y1, x2, y2, conf])

    detections = np.array(detections)

    # ---------- SORT ----------
    if len(detections):
        tracks = tracker.update(detections)
    else:
        tracks = []

    # ---------- DRAW + COUNT ----------
    for track in tracks:
        x1, y1, x2, y2, track_id = map(int, track)
        cy = (y1 + y2) // 2

        if cy > line_y and track_id not in counted_ids:
            count += 1
            counted_ids.add(track_id)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID {track_id}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.line(frame, (0, line_y), (width, line_y), (0, 0, 255), 2)
    cv2.putText(frame, f"Count: {count}", (40, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    out.write(frame)

cap.release()
out.release()

print("✅ Finished!")
print(f"📹 Output video: {out_path}")
print(f"🔢 Total count: {count}")

✅ SORT loaded via manual import
✅ YOLO model loaded
▶️ Processing video...
✅ Finished!
📹 Output video: /content/counted_output.mp4
🔢 Total count: 3


In [ ]:

from google.colab import files
files.download('/content/counted_output.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

!pip install -q ultralytics opencv-python

In [ ]:

# ================== IMPORTS ==================
import cv2
import numpy as np
from ultralytics import YOLO
import importlib.util

# ================== LOAD SORT ==================
spec = importlib.util.spec_from_file_location(
    "sort_module",
    "/content/yolov5/sort/sort.py"
)

sort_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sort_module)

tracker = sort_module.Sort()

print("✅ SORT loaded")

# ================== LOAD YOLO ==================
weights_path = "/content/drive/MyDrive/walnut/runs/detect/train4/weights/best.pt"
model = YOLO(weights_path)

print("✅ YOLO model loaded")

# ================== VIDEO ==================
video_path = "/content/drive/MyDrive/walnut/Video/walnut Tree.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("❌ Video not opened")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25

out_path = "/content/counted_output.mp4"

out = cv2.VideoWriter(
    out_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# ================== COUNTING ==================
count = 0
counted_ids = set()

print("▶️ Processing video...")

# ================== LOOP ==================
while True:

    ret, frame = cap.read()

    if not ret:
        break

    # ---------- YOLO inference ----------
    results = model(frame, conf=0.1, iou=0.5, verbose=False)[0]

    detections = []

    if results.boxes is not None:
        for box in results.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            detections.append([x1, y1, x2, y2, conf])

    detections = np.array(detections)

    # ---------- SORT tracking ----------
    if len(detections):
        tracks = tracker.update(detections)
    else:
        tracks = []

    # ---------- DRAW + COUNT ----------
    for track in tracks:

        x1, y1, x2, y2, track_id = map(int, track)

        # اگر این ID قبلاً شمرده نشده
        if track_id not in counted_ids:
            counted_ids.add(track_id)
            count += 1

        # رسم باکس
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0,255,0),
            2
        )

        cv2.putText(
            frame,
            f"ID {track_id}",
            (x1, y1-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,255,0),
            2
        )

    # نمایش شمارش
    cv2.putText(
        frame,
        f"Count: {count}",
        (40,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,0,255),
        2
    )

    out.write(frame)

cap.release()
out.release()

print("✅ Finished!")
print("📹 Output video:", out_path)
print("🔢 Total count:", count)

✅ SORT loaded
✅ YOLO model loaded
▶️ Processing video...
✅ Finished!
📹 Output video: /content/counted_output.mp4
🔢 Total count: 3
